In [4]:
# =========================================
# Machine Learning Aula: Preparação + Regressão Linear e Logística
# Dataset: Students Performance in Exams (Kaggle)
# =========================================

# 1) Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, classification_report

# 2) Leitura
df = pd.read_csv("StudentsPerformance.csv")

# 3) Inspeção inicial
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

# 4) Limpeza básica
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Verifica valores ausentes
print("\nValores nulos por coluna:\n", df.isna().sum())

# Se houver valores faltantes, substitui:
df = df.fillna({
    'reading_score': df['reading_score'].median(),
    'writing_score': df['writing_score'].median(),
    'math_score': df['math_score'].median()
})

# 5) Criação de variável binária (para logística)
df["pass_math"] = (df["math_score"] >= 60).astype(int)

# 6) Identificação dos tipos de variáveis
num_features = ["reading_score", "writing_score"]
cat_features = ["gender", "race/ethnicity", "parental_level_of_education", "lunch", "test_preparation_course"]

# 7) Separação de variáveis-alvo
y_reg = df["math_score"]
y_clf = df["pass_math"]

# 8) Pipeline de pré-processamento
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ]
)

# 9) Split treino/teste
X = df[num_features + cat_features]
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
X_train2, X_test2, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

# 10) Modelo Linear (com pipeline)
lin_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lin_model.fit(X_train, y_train_reg)
y_pred = lin_model.predict(X_test)

print("\n--- RESULTADOS REGRESSÃO LINEAR ---")
print("R²:", r2_score(y_test_reg, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_reg, y_pred)))

# 11) Modelo Logístico (com pipeline)
log_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

log_model.fit(X_train2, y_train_clf)
y_pred_log = log_model.predict(X_test2)

print("\n--- RESULTADOS REGRESSÃO LOGÍSTICA ---")
print(confusion_matrix(y_test_clf, y_pred_log))
print(classification_report(y_test_clf, y_pred_log))


(1000, 8)
   gender race/ethnicity parental level of education         lunch  \
0  female        group B           bachelor's degree      standard   
1  female        group C                some college      standard   
2  female        group B             master's degree      standard   
3    male        group A          associate's degree  free/reduced   
4    male        group C                some college      standard   

  test preparation course  math score  reading score  writing score  
0                    none          72             72             74  
1               completed          69             90             88  
2                    none          90             95             93  
3                    none          47             57             44  
4                    none          76             78             75  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null C